# Vera seizure monitor — evaluation on the CHB-MIT Scalp EEG Database (v2: band-limited front-end)

This notebook replays public, expert-annotated EEG through **both of Vera's seizure detectors exactly as shipped**
and scores them. Everything runs on Kaggle's servers; nothing heavy happens on your own computer.

* **Python analyzer** (`seizure_monitor.py`, pulled unmodified from GitHub) is driven through its own `serial_reader()` loop with a
  simulated serial port and a simulated clock. Its warm-up and re-adaptation timing match live use.
* **ESP32 firmware detector** (`loop()` in the `.ino`) is a line-by-line port, JIT-compiled.
* **Front-end model:** EEG in µV → 0.5–40 Hz analog pass-band → 200 Hz → ×1000 (EXG BioAmp Pill) → 12-bit / 3.3 V ADC around mid-rail.
* **Data:** all 141 CHB-MIT recordings that contain seizures (24 cases, 198 seizures), plus seizure-free recordings, for Vera's own FP1–FP2 montage
  (reconstructed exactly from a chain of CHB-MIT bipolar channels) plus FP1-F7 and FP2-F8 as secondary channels.
* **Scoring:** event-based. An alarm within [onset − 30 s, offset + 60 s] counts as a detection. Alarms closer than 90 s are merged.
  Every other alarm is a false alarm. A chance-level baseline is reported next to each detector.
* **Proposed revision:** a line-length-ratio detector, with parameters chosen by patient-wise 2-fold cross-validation.

### How to run
1. In the editor's right-hand panel (**Session options**), switch **Internet → On**. If the switch is greyed out, verify your phone number at kaggle.com/settings first. Accelerator: **None** (CPU is enough).
2. Click **Save Version → Save & Run All (Commit)**. It runs in the background (about 4–5 h), and you can close the tab.
3. When it finishes, open the version → **Output**, and download **`vera_results.zip`** (a few MB).

*Optional:* if you attach a CHB-MIT dataset to the notebook, it is used instead of downloading.

In [ ]:
# ── configuration ────────────────────────────────────────────────────────────
REPO        = "https://github.com/tsharan17/Epilepsy-Seizure-Monitoring-System.git"
CHANNELS    = ["FP1-FP2", "FP1-F7", "FP2-F8"]   # FP1-FP2 = Vera's own montage (primary)
PRIMARY     = "FP1-FP2"
GAIN        = 1000.0              # EXG BioAmp Pill gain assumed in the main analysis
BAND        = "0.5,40"            # EXG BioAmp Pill pass-band (Hz) per the Vera circuit diagram; "none" = unfiltered
GAIN_SWEEP  = [500.0, 2000.0]     # robustness check (FP1-F7 only); set [] to skip
FETCH_ALL   = False               # True = every CHB-MIT recording (~42 GB) instead of seizure files + 1 seizure-free / case
import os
WORK        = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATA        = os.environ.get("VERA_DATA", f"{WORK}/data/chbmit")     # extracted 2-channel files
RESULTS     = f"{WORK}/results"
JOBS        = os.cpu_count() or 2
print(WORK, DATA, RESULTS, "cpus:", JOBS)

In [ ]:
# ── internet check: this notebook downloads the code (GitHub) and EEG (PhysioNet's AWS mirror) ──
import socket, os
def _online(host):
    try: socket.create_connection((host, 443), timeout=8).close(); return True
    except OSError: return False
if not os.environ.get("VERA_DATA"):
    missing = [h for h in ("github.com", "physionet-open.s3.amazonaws.com") if not _online(h)]
    if missing:
        raise SystemExit(
            "\n*** No internet access from this notebook (cannot reach: " + ", ".join(missing) + ") ***\n"
            "Fix: open the notebook editor -> right-hand panel 'Session options' -> Internet -> ON,\n"
            "then Save Version -> Save & Run All again.  If the Internet switch is greyed out,\n"
            "verify your phone number at kaggle.com/settings (Kaggle requires it for internet).")
    print("internet OK")

In [ ]:
import subprocess, sys, json, time, platform, shutil
from pathlib import Path
def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True)
    if r.returncode: raise RuntimeError(f"failed ({r.returncode}): {cmd}")

for pkg in ["numba", "pandas", "scipy", "requests", "matplotlib"]:   # all preinstalled on Kaggle
    try: __import__(pkg)
    except ImportError: sh(f"{sys.executable} -m pip install -q {pkg}")

repo = Path(WORK) / "vera"
if not (repo / "seizure_monitor.py").exists():
    if os.environ.get("VERA_MONITOR"):            # local testing hook
        repo.mkdir(parents=True, exist_ok=True); shutil.copy(os.environ["VERA_MONITOR"], repo / "seizure_monitor.py")
    else:
        sh(f"git clone --depth 1 {REPO} {repo}")
commit = subprocess.run(f"git -C {repo} rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip() or "local"
MONITOR = str(repo / "seizure_monitor.py")
print("seizure_monitor.py @", commit)
Path(f"{WORK}/evaluation").mkdir(exist_ok=True)

In [ ]:
%%writefile {WORK}/evaluation/chbmit.py
"""
CHB-MIT loading + Vera front-end simulation.

Data files are produced by `fetch_chbmit.js` (run in a browser on the PhysioNet
S3 mirror): for every EDF it stores only the FP1-F7 / FP2-F8 channels as raw
little-endian int16 digital values (`<rec>.ch.bin`, interleaved [n, n_ch]) plus
the EDF calibration header (`<rec>.ch.json`).  Nothing else is altered.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from scipy.signal import resample_poly

# ── Vera analog front-end model (see README / PIN_CONNECTIONS) ───────────────
FRONTEND_GAIN = 1000.0      # EXG BioAmp Pill gain
ADC_VREF = 3.3              # ESP32 ADC full scale (V), 12-bit
ADC_BITS = 12
ADC_MID = 2 ** (ADC_BITS - 1)   # single-supply mid-rail bias -> 2048
DEVICE_FS = 200             # firmware samples at ~200 Hz (delay(5))


@dataclass
class Record:
    case: str
    name: str
    fs: float
    uv: dict          # channel label -> float32 µV array at native fs
    seizures: list    # [(onset_s, offset_s), ...]

    @property
    def duration_s(self) -> float:
        return len(next(iter(self.uv.values()))) / self.fs


def parse_summary(path: Path) -> dict:
    """Return {edf_name: [(onset, offset), ...]} for every file in a summary."""
    txt = Path(path).read_text(errors="replace")
    out = {}
    for block in re.split(r"File Name:\s*", txt)[1:]:
        name = block.split()[0].strip()
        starts = [int(x) for x in re.findall(r"Seizure\s*\d*\s*Start Time:\s*(\d+)", block)]
        ends = [int(x) for x in re.findall(r"Seizure\s*\d*\s*End Time:\s*(\d+)", block)]
        n = re.search(r"Number of Seizures in File:\s*(\d+)", block)
        n = int(n.group(1)) if n else len(starts)
        if len(starts) != n or len(ends) != n:
            raise ValueError(f"{path}: seizure count mismatch for {name}")
        out[name] = list(zip(starts, ends))
    return out


def load_record(data_dir: Path, case: str, name: str, seizures) -> Record:
    meta = json.loads((data_dir / f"{name}.ch.json").read_text())
    raw = np.fromfile(data_dir / f"{name}.ch.bin", dtype="<i2")
    nch = len(meta["labels"])
    raw = raw.reshape(-1, nch).astype(np.float64)
    uv = {}
    for i, lab in enumerate(meta["labels"]):
        pmin, pmax = meta["phys_min"][i], meta["phys_max"][i]
        dmin, dmax = meta["dig_min"][i], meta["dig_max"][i]
        gain = (pmax - pmin) / (dmax - dmin)
        x = (raw[:, i] - dmin) * gain + pmin
        unit = meta["dim"][i].lower()
        if unit in ("mv",):
            x = x * 1e3
        elif unit in ("v",):
            x = x * 1e6
        uv[lab.upper()] = x.astype(np.float32)
    return Record(case, name, float(meta["fs"]), uv, list(seizures))


FRONTEND_BAND = (0.5, 40.0)   # EXG BioAmp Pill pass-band per the Vera circuit diagram


def frontend_filter(uv: np.ndarray, fs: float, band=FRONTEND_BAND) -> np.ndarray:
    """Causal model of the analog pass-band: 1st-order HP + 2nd-order Butterworth LP."""
    if band is None:
        return uv
    from scipy.signal import butter, lfilter
    lo, hi = band
    b, a = butter(1, lo, btype="highpass", fs=fs)
    y = lfilter(b, a, uv)
    b, a = butter(2, hi, btype="lowpass", fs=fs)
    return lfilter(b, a, y)


def to_device_adc(uv: np.ndarray, fs_in: float, gain: float = FRONTEND_GAIN,
                  band=FRONTEND_BAND) -> np.ndarray:
    """µV at fs_in  ->  integer ESP32 ADC codes at 200 Hz, as Vera would see them.

    analog pass-band (optional) -> resample (polyphase, anti-aliased) -> ×gain
    -> +mid-rail -> quantise -> clip.
    """
    from fractions import Fraction
    x = frontend_filter(uv.astype(np.float64), fs_in, band)
    fr = Fraction(DEVICE_FS / fs_in).limit_denominator(1000)
    x = resample_poly(x, fr.numerator, fr.denominator)
    volts = x * 1e-6 * gain
    codes = np.round(ADC_MID + volts / (ADC_VREF / 2 ** ADC_BITS))
    return np.clip(codes, 0, 2 ** ADC_BITS - 1).astype(np.int32)


def discover(data_root: Path):
    """Yield (case, edf_name, seizures, data_dir) for every extracted record."""
    data_root = Path(data_root)
    for summ in sorted(data_root.glob("chb*-summary.txt")):
        case = summ.name.split("-")[0]
        table = parse_summary(summ)
        for edf, sz in table.items():
            name = edf.replace(".edf", "")
            if (data_root / f"{name}.ch.json").exists():
                yield case, name, sz, data_root


In [ ]:
%%writefile {WORK}/evaluation/detectors.py
"""
Offline replay of Vera's two detectors.

1. `run_python_detector` drives the *unmodified* seizure_monitor.py through its
   own `serial_reader()` loop: a fake serial port emits the firmware's
   "eeg baseline diff state thr" lines and a simulated clock advances 5 ms per
   line, so warm-up / re-adaptation timing is exactly what it would be live.
   Only I/O side-effects (alerts, JSONL log, plotting backend) are stubbed.

2. `run_firmware_detector` is a line-by-line port of loop() in
   seizure_monitor_esp32_firmware.ino (EMA baseline, |x-baseline| diff, Welford
   EWMA, adaptive μ+Kσ thresholds, confirm counters, post-ictal hold), JIT
   compiled with numba.  C integer casts/truncation are reproduced.

Both return a per-tick state trace: (times_s, states) with states coded
0=NORMAL 1=BROWNOUT 2=BLACKOUT 3=POST-ICTAL.
"""
from __future__ import annotations

import importlib.util
import logging
import sys
import types
from pathlib import Path

import numpy as np
from numba import njit

STATE_CODE = {"NORMAL": 0, "BROWNOUT": 1, "BLACKOUT": 2, "POST-ICTAL": 3}
FS = 200.0
DT = 1.0 / FS


# ═════════════════════════════════════════════════════════════════════════════
#  Python analyzer (seizure_monitor.py) — unmodified module, replayed
# ═════════════════════════════════════════════════════════════════════════════
class _SimClock:
    def __init__(self):
        self.t = 1_000_000.0

    def time(self):
        return self.t

    def sleep(self, _):
        pass

    def __getattr__(self, item):            # monotonic, perf_counter, …
        import time as _t
        return getattr(_t, item)


def _ensure_serial_module():
    """seizure_monitor.py imports pyserial; replay never opens a port, so a stub suffices."""
    try:
        import serial  # noqa: F401
    except ImportError:
        stub = types.ModuleType("serial")

        class SerialException(Exception):
            pass

        class Serial:  # pragma: no cover - never instantiated during replay
            def __init__(self, *a, **k):
                raise SerialException("pyserial stub: no real ports in offline replay")

        stub.Serial, stub.SerialException = Serial, SerialException
        sys.modules["serial"] = stub


def load_monitor_module(path: Path):
    _ensure_serial_module()
    import matplotlib
    matplotlib.use = lambda *a, **k: None   # TkAgg not needed headless
    matplotlib.rcParams["backend"] = "Agg"
    spec = importlib.util.spec_from_file_location("seizure_monitor", str(path))
    mod = importlib.util.module_from_spec(spec)
    sys.modules["seizure_monitor"] = mod
    spec.loader.exec_module(mod)
    logging.getLogger(mod.__name__).setLevel(logging.ERROR)
    logging.getLogger().setLevel(logging.ERROR)
    mod.log_event = lambda *a, **k: None
    mod.send_alert = lambda *a, **k: None
    return mod


class _FakeSerial:
    """Emits firmware-format serial lines and advances the simulated clock."""

    def __init__(self, mod, clock, adc, fw_baseline, fw_diff, fw_state, fw_thr):
        self.mod, self.clock = mod, clock
        self.adc, self.base, self.diff = adc, fw_baseline, fw_diff
        self.state, self.thr = fw_state, fw_thr
        self.i, self.n = 0, len(adc)

    def readline(self):
        if self.i >= self.n:
            self.mod.SHARED.running = False
            return b""
        i = self.i
        self.i += 1
        self.clock.t += DT
        return (f"{self.adc[i]} {self.base[i]:.2f} {self.diff[i]} "
                f"{self.state[i]} {self.thr[i]}\n").encode()


def run_python_detector(mod, adc, fw_trace, cfg_overrides=None):
    """Replay one recording through seizure_monitor.serial_reader()."""
    from collections import deque

    clock = _SimClock()
    mod.time = clock                                   # module-level `time`
    mod.CFG = mod.Config(**(cfg_overrides or {}))
    mod.ADAPT = mod.AdaptiveThresholds()               # start = sim time 0
    mod.SHARED = mod.SharedState()
    mod._prev_seizure_state = "NORMAL"
    mod._confirm_buf = deque(maxlen=mod.CFG.confirm_ticks)
    # `update_seizure_state` reads _confirm_buf as a module global → rebinding works

    times, states, feats, thr = [], [], [], []
    orig = getattr(mod.update_seizure_state, "__wrapped__", mod.update_seizure_state)
    orig_an = getattr(mod.analyse, "__wrapped__", mod.analyse)

    def traced(raw_class, alpha_r, beta_r, ll, dar, hfe):
        s = orig(raw_class, alpha_r, beta_r, ll, dar, hfe)
        times.append(clock.t - 1_000_000.0)
        states.append(STATE_CODE[s])
        if len(states) % 10 == 1:                      # thresholds at ~1 Hz
            A = mod.ADAPT
            thr.append((clock.t - 1_000_000.0, A.ll_brownout, A.dar_brownout, A.hfe_brownout,
                        A.ll_blackout, A.dar_blackout, A.hfe_blackout))
        return s

    def traced_an(samples):                            # every tick, incl. artifacts
        r = orig_an(samples)
        feats.append((clock.t - 1_000_000.0, r[4], r[5], r[6]))
        return r

    traced.__wrapped__ = orig; traced_an.__wrapped__ = orig_an
    mod.update_seizure_state = traced; mod.analyse = traced_an

    base, diff, st, thr_fw = fw_trace
    ser = _FakeSerial(mod, clock, adc, base, diff, st, thr_fw)
    try:
        mod.serial_reader(ser)
    finally:
        mod.update_seizure_state = orig; mod.analyse = orig_an
    return (np.asarray(times), np.asarray(states, dtype=np.int8), mod.SHARED.artifact_count,
            np.asarray(feats, dtype=np.float32).reshape(-1, 4),
            np.asarray(thr, dtype=np.float32).reshape(-1, 7))


# ═════════════════════════════════════════════════════════════════════════════
#  ESP32 firmware detector — faithful port of loop()
# ═════════════════════════════════════════════════════════════════════════════
FW = dict(
    EMA_ALPHA=0.01, ADAPT_WARMUP_MS=90000, ADAPT_INTERVAL_MS=30000,
    Z_B=2.5, Z_K=4.0, B_FLOOR=60, K_FLOOR=120, B_CEIL=1200, K_CEIL=2400,
    WF_FORGET=0.995, CONFIRM_B=20, CONFIRM_K=15, POSTICTAL_HOLD=400,
    SAMPLE_MS=5,
)


@njit(cache=True)
def _constrain(x, lo, hi):
    return lo if x < lo else (hi if x > hi else x)


@njit(cache=True)
def _fw_loop(adc, ema_a, warm_ms, adapt_ms, zb, zk, bf, kf, bc, kc, lam,
             cb, ck, hold, sample_ms):
    n = adc.shape[0]
    # setup(): baseline = integer mean of first 200 samples (long sum / 200)
    ncal = 200 if n > 200 else n
    s = 0
    for i in range(ncal):
        s += adc[i]
    baseline = np.float32(s // ncal)
    # Welford seed
    wn = 0.0; wm = 0.0; wM2 = 0.0
    for _ in range(20):
        wn = lam * wn + 1.0
        d = 10.0 - wm
        wm += d / wn
        wM2 = lam * wM2 + d * (10.0 - wm)
    thrB = 80; thrK = 160
    state = 0; cB = 0; cK = 0; pcount = 0
    start_ms = ncal * sample_ms
    last_adapt = start_ms
    out_base = np.empty(n - ncal, np.float32)
    out_diff = np.empty(n - ncal, np.int32)
    out_state = np.empty(n - ncal, np.int8)
    out_thr = np.empty(n - ncal, np.int32)
    one_m = np.float32(1.0) - np.float32(ema_a)
    a32 = np.float32(ema_a)
    for j in range(n - ncal):
        i = j + ncal
        now = i * sample_ms
        x = adc[i]
        baseline = one_m * baseline + a32 * np.float32(x)
        diff = abs(x - int(baseline))
        if state == 0:
            xf = np.float64(np.float32(diff))
            wn = lam * wn + 1.0
            d = xf - wm
            wm += d / wn
            wM2 = lam * wM2 + d * (xf - wm)
        if (now - start_ms) >= warm_ms and (now - last_adapt) >= adapt_ms:
            mu = np.float32(wm)
            sig = np.float32(0.0) if wn < 2.0 else np.float32(np.sqrt(wM2 / wn))
            if sig >= 1.0:
                nb = _constrain(int(mu + np.float32(zb) * sig), bf, bc)
                nk = _constrain(int(mu + np.float32(zk) * sig), kf, kc)
                if nk <= nb:
                    nk = nb + 40
                thrB = nb; thrK = nk
            last_adapt = now
        if state == 0:
            if diff >= thrK:
                cK += 1
                if cK >= ck:
                    state = 2; cK = 0; cB = 0
            elif diff >= thrB:
                cK = 0
                cB += 1
                if cB >= cb:
                    state = 1; cB = 0
            else:
                if cB > 0: cB -= 1
                if cK > 0: cK -= 1
        elif state == 1:
            if diff >= thrK:
                cK += 1
                if cK >= ck:
                    state = 2; cK = 0
            elif diff < thrB:
                cB += 1
                if cB >= cb:
                    state = 0; cB = 0
            else:
                cB = 0; cK = 0
        elif state == 2:
            if diff < thrB:
                state = 3; pcount = hold
        else:
            pcount -= 1
            if pcount <= 0 and diff < thrB:
                state = 0
        out_base[j] = baseline; out_diff[j] = diff
        out_state[j] = state; out_thr[j] = thrB
    return ncal, out_base, out_diff, out_state, out_thr


def run_firmware_detector(adc, **over):
    p = dict(FW, **over)
    ncal, base, diff, st, thr = _fw_loop(
        np.ascontiguousarray(adc, dtype=np.int32), p["EMA_ALPHA"],
        p["ADAPT_WARMUP_MS"], p["ADAPT_INTERVAL_MS"], p["Z_B"], p["Z_K"],
        p["B_FLOOR"], p["K_FLOOR"], p["B_CEIL"], p["K_CEIL"], p["WF_FORGET"],
        p["CONFIRM_B"], p["CONFIRM_K"], p["POSTICTAL_HOLD"], p["SAMPLE_MS"])
    t = (np.arange(len(st)) + ncal) * DT
    return ncal, t, st, (base, diff, st, thr)


In [ ]:
%%writefile {WORK}/evaluation/scoring.py
"""
Event-based scoring (SzCORE-style any-overlap with tolerances).

An *alarm* is the moment the detector enters an alarm state from a non-alarm
state.  Two alarm definitions are reported:
  • "seizure alarm"  : entry into BLACKOUT                (the urgent alert)
  • "any alarm"      : entry into BROWNOUT or BLACKOUT     (warning or urgent)
Alarms closer than MERGE_S to the previous alarm are merged (one alert).

A seizure is detected if an alarm falls in [onset − PRE_S, offset + POST_S].
Latency = first such alarm − onset (negative = before annotated onset).
Every alarm that falls in no seizure's window is a false alarm.
"""
from __future__ import annotations

import numpy as np

PRE_S = 30.0
POST_S = 60.0
MERGE_S = 90.0


def alarm_times(t, states, kind="seizure"):
    t = np.asarray(t); s = np.asarray(states)
    alarm = (s == 2) if kind == "seizure" else ((s == 1) | (s == 2))
    prev = np.concatenate([[False], alarm[:-1]])
    onsets = t[alarm & ~prev]
    merged = []
    for a in onsets:
        if not merged or a - merged[-1] >= MERGE_S:
            merged.append(float(a))
    return merged


def score_record(alarms, seizures, duration_s):
    det, lat = [], []
    used = set()
    for (on, off) in seizures:
        hits = [a for a in alarms if on - PRE_S <= a <= off + POST_S]
        det.append(bool(hits))
        lat.append(min(hits) - on if hits else np.nan)
        used.update(hits)
    fa = [a for a in alarms if a not in used]
    return dict(n_sz=len(seizures), detected=det, latency=lat,
                n_fa=len(fa), fa_times=fa, hours=duration_s / 3600.0)


def summarise(rows):
    """rows: list of dicts from score_record (+ 'case').  Pooled + per-case."""
    n_sz = sum(r["n_sz"] for r in rows)
    n_det = sum(sum(r["detected"]) for r in rows)
    hours = sum(r["hours"] for r in rows)
    n_fa = sum(r["n_fa"] for r in rows)
    lats = [l for r in rows for l in r["latency"] if not np.isnan(l)]
    per_case = {}
    for r in rows:
        c = per_case.setdefault(r["case"], dict(n_sz=0, n_det=0, n_fa=0, hours=0.0, lat=[]))
        c["n_sz"] += r["n_sz"]; c["n_det"] += sum(r["detected"])
        c["n_fa"] += r["n_fa"]; c["hours"] += r["hours"]
        c["lat"] += [l for l in r["latency"] if not np.isnan(l)]
    case_sens = [c["n_det"] / c["n_sz"] for c in per_case.values() if c["n_sz"]]
    return dict(
        seizures=n_sz, detected=n_det,
        sensitivity=n_det / n_sz if n_sz else np.nan,
        mean_case_sensitivity=float(np.mean(case_sens)) if case_sens else np.nan,
        hours=hours, false_alarms=n_fa, fa_per_hour=n_fa / hours if hours else np.nan,
        fa_per_24h=24 * n_fa / hours if hours else np.nan,
        latency_median=float(np.median(lats)) if lats else np.nan,
        latency_iqr=(float(np.percentile(lats, 25)), float(np.percentile(lats, 75))) if lats else (np.nan, np.nan),
        per_case=per_case,
    )


In [ ]:
%%writefile {WORK}/evaluation/fetch_chbmit.py
"""
Fetch CHB-MIT recordings and keep only the channels Vera can see.

For every EDF: download (PhysioNet AWS open-data mirror, or read from a local /
attached copy), extract FP1-F7 and FP2-F8 as raw int16 + calibration header
(`<rec>.ch.bin` / `<rec>.ch.json`), then delete the EDF.  If a recording uses a
referential montage (e.g. some chb12 files), the bipolar channel is derived as
FP1-ref − F7-ref and flagged `derived` in the json.

Selection (default): every recording that contains a seizure (141 files,
198 seizures) + the first seizure-free recording of every patient
(+ chb01_01 and chb01_02) for additional false-alarm hours.

usage: python fetch_chbmit.py --out data/chbmit [--local /kaggle/input/...] [--workers 6]
"""
from __future__ import annotations

import argparse
import json
import os
import re
import sys
import time
import urllib.parse
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np

S3 = "https://physionet-open.s3.amazonaws.com/chbmit/1.0.0/"
WANT = {"FP1-F7": ("FP1", "F7"), "FP2-F8": ("FP2", "F8")}
CASES = [f"chb{i:02d}" for i in range(1, 25)]


def _get(url, tries=4):
    import requests
    for k in range(tries):
        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            return r.content
        except Exception:
            if k == tries - 1:
                raise
            time.sleep(2 * (k + 1))


def read_edf_channels(buf: bytes):
    """Minimal EDF reader → (meta, {label: int16 array})."""
    h = lambda o, l: buf[o:o + l].decode("ascii", "replace").strip()
    nrec, dur, ns = int(h(236, 8)), float(h(244, 8)), int(h(252, 4))
    o = 256
    def field(w):
        nonlocal o
        v = [h(o + i * w, w) for i in range(ns)]; o += ns * w; return v
    labels = field(16); field(80); dim = field(8)
    pmin = list(map(float, field(8))); pmax = list(map(float, field(8)))
    dmin = list(map(float, field(8))); dmax = list(map(float, field(8)))
    field(80); nsamp = list(map(int, field(8))); field(32)
    hdr = 256 + ns * 256
    rec_len = sum(nsamp)
    n_real = (len(buf) - hdr) // (rec_len * 2)
    nrec = min(nrec if nrec > 0 else n_real, n_real)
    data = np.frombuffer(buf, dtype="<i2", count=nrec * rec_len, offset=hdr).reshape(nrec, rec_len)
    offs = np.concatenate([[0], np.cumsum(nsamp)[:-1]])
    def chan(i):
        return data[:, offs[i]:offs[i] + nsamp[i]].reshape(-1)
    return dict(labels=labels, dim=dim, pmin=pmin, pmax=pmax, dmin=dmin, dmax=dmax,
                nsamp=nsamp, dur=dur), chan


def _bipolar_path(labels_upper, a, b):
    """Shortest chain of bipolar channels from electrode a to b → [(index, ±1), ...]."""
    from collections import deque
    adj = {}
    for k, l in enumerate(labels_upper):
        parts = l.split("-")
        if len(parts) != 2 or not parts[0] or not parts[1]:
            continue
        x, y = parts
        adj.setdefault(x, []).append((y, k, +1))     # x - y : stepping x→y adds +(x−y)
        adj.setdefault(y, []).append((x, k, -1))     # stepping y→x adds −(x−y)
    prev = {a: None}
    q = deque([a])
    while q:
        u = q.popleft()
        if u == b:
            break
        for v, k, s in adj.get(u, []):
            if v not in prev:
                prev[v] = (u, k, s); q.append(v)
    if b not in prev:
        return None
    path, node = [], b
    while prev[node] is not None:
        u, k, s = prev[node]; path.append((k, s)); node = u
    return path[::-1]


def extract(buf: bytes, source: str, out_dir: Path, name: str):
    H, chan = read_edf_channels(buf)
    # plain electrode names (e.g. chb12_28: "FP1", "F7", … against a common
    # reference) are treated as "<name>-REF" so referential montages work too
    up = [l.upper() if ("-" in l or not l.strip()) else l.upper().strip() + "-REF" for l in H["labels"]]
    cols, meta = [], dict(source=source, labels=[], dim=[], phys_min=[], phys_max=[],
                          dig_min=[], dig_max=[], derived=[], missing=[], all_labels=H["labels"])
    for want, (a, b) in WANT.items():
        if want in up:
            i = up.index(want)
            cols.append(chan(i).astype(np.int16))
            meta["labels"].append(want); meta["dim"].append(H["dim"][i])
            meta["phys_min"].append(H["pmin"][i]); meta["phys_max"].append(H["pmax"][i])
            meta["dig_min"].append(H["dmin"][i]); meta["dig_max"].append(H["dmax"][i])
            meta["derived"].append(False)
            fs = H["nsamp"][i] / H["dur"]
            continue
        # referential fallback: "FP1-<ref>" and "F7-<ref>" with the same ref
        ia = [k for k, l in enumerate(up) if l.split("-")[0] == a and "-" in l]
        ib = [k for k, l in enumerate(up) if l.split("-")[0] == b and "-" in l]
        pair = next(((i, j) for i in ia for j in ib if up[i].split("-", 1)[1] == up[j].split("-", 1)[1]), None)
        if pair is None:
            meta["missing"].append(want); continue
        i, j = pair
        # convert both to physical units, subtract, re-quantise on channel i's scale
        g = lambda k: (H["pmax"][k] - H["pmin"][k]) / (H["dmax"][k] - H["dmin"][k])
        phys = lambda k: (chan(k).astype(np.float64) - H["dmin"][k]) * g(k) + H["pmin"][k]
        d = phys(i) - phys(j)
        q = np.clip(np.round((d - H["pmin"][i]) / g(i) + H["dmin"][i]), -32768, 32767).astype(np.int16)
        cols.append(q)
        meta["labels"].append(want); meta["dim"].append(H["dim"][i])
        meta["phys_min"].append(H["pmin"][i]); meta["phys_max"].append(H["pmax"][i])
        meta["dig_min"].append(H["dmin"][i]); meta["dig_max"].append(H["dmax"][i])
        meta["derived"].append(f"{H['labels'][i]} - {H['labels'][j]}")
        fs = H["nsamp"][i] / H["dur"]
    # Vera's own montage: FP1 − FP2, reconstructed exactly as a signed sum of a
    # chain of bipolar channels (e.g. FP1-F7, F7-T7, T7-FT9, FT9-FT10, FT10-T8, F8-T8, FP2-F8)
    chain = _bipolar_path(up, "FP1", "FP2")
    if chain:
        g = lambda k: (H["pmax"][k] - H["pmin"][k]) / (H["dmax"][k] - H["dmin"][k])
        phys = lambda k: (chan(k).astype(np.float64) - H["dmin"][k]) * g(k) + H["pmin"][k]
        n0 = min(len(chan(k)) for k, _ in chain)
        d = sum(sgn * phys(k)[:n0] for k, sgn in chain)
        # store at 0.2 µV/LSB (±6553.4 µV) — finer than CHB-MIT's native resolution
        q = np.clip(np.round(d / 0.2), -32767, 32767).astype(np.int16)
        meta["fp1fp2_clip_frac"] = float(np.mean(np.abs(d / 0.2) > 32767))
        cols.append(q)
        meta["labels"].append("FP1-FP2"); meta["dim"].append("uV")
        meta["phys_min"].append(-6553.4); meta["phys_max"].append(6553.4)
        meta["dig_min"].append(-32767); meta["dig_max"].append(32767)
        meta["derived"].append(" + ".join(("" if s > 0 else "−") + H["labels"][k] for k, s in chain))
        fs = H["nsamp"][chain[0][0]] / H["dur"]
    else:
        meta["missing"].append("FP1-FP2")
    if not cols:
        return dict(name=name, error="no usable frontal channels", labels=H["labels"])
    n = min(len(c) for c in cols)
    arr = np.stack([c[:n] for c in cols], 1)
    meta.update(fs=fs, n=int(n))
    arr.astype("<i2").tofile(out_dir / f"{name}.ch.bin")
    (out_dir / f"{name}.ch.json").write_text(json.dumps(meta))
    return dict(name=name, n=int(n), fs=fs, labels=meta["labels"], derived=meta["derived"])


def select_files(summaries: dict) -> list:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
    from chbmit import parse_summary
    files = []
    for case in CASES:
        p = summaries[case]
        table = parse_summary(p)
        sz = [f for f, s in table.items() if s]
        free = [f for f, s in table.items() if not s]
        extra = free[:2] if case == "chb01" else free[:1]
        files += [f"{case}/{f}" for f in sz + extra]
    return files


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    ap.add_argument("--local", default=None, help="folder holding chbXX/ EDFs (e.g. an attached Kaggle dataset)")
    ap.add_argument("--workers", type=int, default=6)
    ap.add_argument("--all", action="store_true", help="every recording (≈42 GB download)")
    a = ap.parse_args()
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)

    local = None
    if a.local:
        hits = list(Path(a.local).rglob("chb01-summary.txt"))
        local = hits[0].parent.parent if hits else None
        print("local copy:", local)

    summaries = {}
    for c in CASES:
        dst = out / f"{c}-summary.txt"
        if not dst.exists():
            src = (local / c / f"{c}-summary.txt") if local else None
            dst.write_bytes(src.read_bytes() if src and src.exists() else _get(S3 + f"{c}/{c}-summary.txt"))
        summaries[c] = dst

    if a.all:
        from chbmit import parse_summary
        files = [f"{c}/{f}" for c in CASES for f in parse_summary(summaries[c])]
    else:
        files = select_files(summaries)
    todo = [f for f in files if not (out / (Path(f).stem + ".ch.json")).exists()]
    print(f"{len(files)} recordings selected, {len(todo)} to fetch", flush=True)

    def job(f):
        name = Path(f).stem
        lp = (local / f) if local else None
        if lp is not None and lp.exists():
            buf = lp.read_bytes()
        else:
            buf = _get(S3 + urllib.parse.quote(f))
        return extract(buf, "chbmit/1.0.0/" + f, out, name)

    log = []
    t0 = time.time()
    with ThreadPoolExecutor(a.workers) as ex:
        futs = {ex.submit(job, f): f for f in todo}
        for i, fu in enumerate(as_completed(futs), 1):
            f = futs[fu]
            try:
                r = fu.result()
            except Exception as e:                       # noqa: BLE001
                r = dict(name=f, error=repr(e))
            log.append(r)
            if i % 10 == 0 or "error" in r:
                print(f"[{i}/{len(todo)}] {f} {r.get('error', '')}  ({time.time()-t0:.0f}s)", flush=True)
    with open(out / "fetch_log.jsonl", "a") as fh:
        for r in log:
            fh.write(json.dumps(r) + "\n")
    errs = [r for r in log if "error" in r]
    print(f"done: {len(log) - len(errs)} ok, {len(errs)} errors")
    for r in errs:
        print("  ", r)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {WORK}/evaluation/run_eval.py
"""
Replay CHB-MIT through both Vera detectors and store per-record results.

usage:
  python run_eval.py --data ../data/chbmit --monitor ../analyzer/seizure_monitor.py \
                     --out results/records.jsonl [--channels FP1-F7 FP2-F8] [--jobs 2]
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
from multiprocessing import Pool
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from chbmit import discover, load_record, to_device_adc          # noqa: E402
from detectors import (load_monitor_module, run_firmware_detector,  # noqa: E402
                       run_python_detector)
from scoring import alarm_times                                    # noqa: E402

_MOD = None
_ARGS = None


def _init(monitor_path, args):
    global _MOD, _ARGS
    _MOD = load_monitor_module(Path(monitor_path))
    _ARGS = args


def _state_occupancy(states):
    s = np.asarray(states)
    return [float(np.mean(s == k)) if len(s) else 0.0 for k in range(4)]


def process(job):
    case, name, seizures, data_dir = job
    t0 = time.time()
    rec = load_record(Path(data_dir), case, name, seizures)
    out = []
    for ch in _ARGS["channels"]:
        if ch not in rec.uv:
            out.append(dict(case=case, record=name, channel=ch, missing=True,
                            seizures=seizures, duration_s=rec.duration_s))
            continue
        adc = to_device_adc(rec.uv[ch], rec.fs, gain=_ARGS["gain"], band=_ARGS["band"])
        clip_frac = float(np.mean((adc == 0) | (adc == 4095)))
        ncal, t_fw, st_fw, trace = run_firmware_detector(adc)
        t_py, st_py, n_art, feats, thr = run_python_detector(_MOD, adc[ncal:], trace)
        t_py = t_py + ncal / 200.0
        feats[:, 0] += ncal / 200.0; thr[:, 0] += ncal / 200.0
        # per-tick feature trace (all analysis ticks, incl. artifact-rejected)
        fdir = Path(_ARGS["feat_dir"]); fdir.mkdir(parents=True, exist_ok=True)
        d = trace[1].astype(np.float32); m = len(d) // 20 * 20     # fw diff @10 Hz
        np.savez_compressed(fdir / f"{name}_{ch}.npz",
                            t=t_py.astype(np.float32), state=st_py,
                            ft=feats[:, 0], ll=feats[:, 1],
                            dar=feats[:, 2], hfe=feats[:, 3], thr=thr,
                            fw_t10=t_fw[:m:20].astype(np.float32),
                            fw_state=st_fw[:m:20],
                            fw_diff10=d[:m].reshape(-1, 20).mean(1))
        art_ceiling = _MOD.Config().artifact_ll_ceiling
        ict = np.zeros(len(feats), bool)
        for on, off in seizures:
            ict |= (feats[:, 0] >= on) & (feats[:, 0] <= off)
        art = feats[:, 1] > art_ceiling
        art_stats = dict(ictal_ticks=int(ict.sum()), ictal_artifact_ticks=int((ict & art).sum()),
                         interictal_ticks=int((~ict).sum()), interictal_artifact_ticks=int((~ict & art).sum()))
        out.append(dict(
            case=case, record=name, channel=ch, missing=False,
            seizures=seizures, duration_s=len(adc) / 200.0,
            adc_clip_frac=clip_frac,
            adc_std=float(np.std(adc)),
            fw=dict(seizure_alarms=alarm_times(t_fw, st_fw, "seizure"),
                    any_alarms=alarm_times(t_fw, st_fw, "any"),
                    occupancy=_state_occupancy(st_fw)),
            py=dict(seizure_alarms=alarm_times(t_py, st_py, "seizure"),
                    any_alarms=alarm_times(t_py, st_py, "any"),
                    occupancy=_state_occupancy(st_py),
                    artifact_epochs=int(n_art), ticks=int(len(st_py)), **art_stats),
            secs=round(time.time() - t0, 1)))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--monitor", required=True)
    ap.add_argument("--out", default="results/records.jsonl")
    ap.add_argument("--channels", nargs="+", default=["FP1-FP2", "FP1-F7", "FP2-F8"])
    ap.add_argument("--jobs", type=int, default=os.cpu_count() or 1)
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--gain", type=float, default=1000.0, help="front-end gain (V/V)")
    ap.add_argument("--band", default="0.5,40", help="front-end pass-band 'lo,hi' in Hz, or 'none'")
    a = ap.parse_args()

    jobs = list(discover(Path(a.data)))
    out_path = Path(a.out); out_path.parent.mkdir(parents=True, exist_ok=True)
    done = set()
    if out_path.exists():
        for line in out_path.read_text().splitlines():
            r = json.loads(line); done.add(r["record"])
    jobs = [j for j in jobs if j[1] not in done]
    if a.limit:
        jobs = jobs[: a.limit]
    print(f"{len(jobs)} records to process ({len(done)} already done)", flush=True)
    args = dict(channels=[c.upper() for c in a.channels],
                feat_dir=str(out_path.parent / "features"), gain=a.gain,
                band=None if a.band.lower() == "none" else tuple(float(v) for v in a.band.split(",")))
    with Pool(a.jobs, initializer=_init, initargs=(a.monitor, args)) as pool, \
            open(out_path, "a") as fh:
        for i, res in enumerate(pool.imap_unordered(process, jobs), 1):
            for r in res:
                fh.write(json.dumps(r) + "\n")
            fh.flush()
            r0 = res[0]
            print(f"[{i}/{len(jobs)}] {r0['record']}  {r0.get('secs', '-')}s", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {WORK}/evaluation/report.py
"""
Aggregate records.jsonl (+ features/*.npz) into paper-ready tables and figures.

usage: python report.py --results results [--data ../data/chbmit]
writes: results/summary.json, results/tables/*.csv|.md, results/figures/*.png
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from scoring import score_record, summarise                     # noqa: E402

DET = {"py": "Python analyzer (LL+DAR+HFE, 2-of-3)", "fw": "ESP32 firmware (abs(x−EMA), μ+Kσ)"}
KIND = {"seizure_alarms": "BLACKOUT alarm", "any_alarms": "BROWNOUT-or-BLACKOUT alarm"}
C_PY, C_FW, INK, MUTED, SZ_FILL = "#2a78d6", "#eb6834", "#1f1f1e", "#6b6a64", "#d9d8d2"


def load(results: Path):
    rows = [json.loads(l) for l in (results / "records.jsonl").read_text().splitlines() if l.strip()]
    return [r for r in rows if not r.get("missing")]


def evaluate(rows, det, kind, channel):
    scored = []
    for r in rows:
        if r["channel"] != channel:
            continue
        s = score_record(r[det][kind], [tuple(x) for x in r["seizures"]], r["duration_s"])
        s["case"] = r["case"]; s["record"] = r["record"]
        scored.append(s)
    return summarise(scored), scored


def chance_sensitivity(rows, channel, fa_per_hour):
    """Expected sensitivity of a Poisson alarm process with the same rate."""
    from scoring import PRE_S, POST_S
    lam = fa_per_hour / 3600.0
    p = [1 - np.exp(-lam * (off - on + PRE_S + POST_S))
         for r in rows if r["channel"] == channel for on, off in r["seizures"]]
    return float(np.mean(p)) if p else np.nan


def fmt_row(name, s):
    lo, hi = s["latency_iqr"]
    return (f"| {name} | {s['detected']}/{s['seizures']} ({100*s['sensitivity']:.1f}%) | "
            f"{100*s['mean_case_sensitivity']:.1f}% | {s['false_alarms']} | {s['fa_per_hour']:.2f} | "
            f"{s['fa_per_24h']:.1f} | {s['latency_median']:.1f} [{lo:.1f}, {hi:.1f}] | {100*s.get('chance', np.nan):.1f}% |")


# ── feature discriminability ────────────────────────────────────────────────
def auc(pos, neg):
    from scipy.stats import mannwhitneyu
    pos, neg = np.asarray(pos), np.asarray(neg)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if len(pos) < 5 or len(neg) < 5:
        return np.nan
    u = mannwhitneyu(pos, neg, alternative="two-sided").statistic
    return float(u / (len(pos) * len(neg)))


def feature_analysis(rows, results: Path, channel):
    """Pooled + per-patient AUC of each feature, ictal vs inter-ictal ticks."""
    fdir = results / "features"
    names = ["Line-length", "Delta/alpha ratio", "High-freq energy", "Firmware abs(x−EMA)"]
    pooled = {n: ([], []) for n in names}
    per_case = {}
    rng = np.random.default_rng(0)
    for r in rows:
        if r["channel"] != channel:
            continue
        f = fdir / f"{r['record']}_{channel}.npz"
        if not f.exists():
            continue
        z = np.load(f)
        t = z["ft"]
        def masks(tt):
            ict = np.zeros(len(tt), bool); far = np.ones(len(tt), bool)
            for on, off in r["seizures"]:
                ict |= (tt >= on) & (tt <= off)
                far &= ~((tt >= on - 300) & (tt <= off + 300))
            far &= tt > 120                                   # skip warm-up
            return ict, far
        ict, far = masks(t)
        fi, ff = masks(z["fw_t10"])
        series = [z["ll"], z["dar"], z["hfe"]]
        for n, x in zip(names[:3], series):
            pooled[n][0].append(x[ict]); pooled[n][1].append(x[far][rng.random(far.sum()) < 0.1])
        pooled[names[3]][0].append(z["fw_diff10"][fi])
        pooled[names[3]][1].append(z["fw_diff10"][ff][rng.random(ff.sum()) < 0.1])
        c = per_case.setdefault(r["case"], {n: ([], []) for n in names})
        for n, x in zip(names[:3], series):
            c[n][0].append(x[ict]); c[n][1].append(x[far])
        c[names[3]][0].append(z["fw_diff10"][fi]); c[names[3]][1].append(z["fw_diff10"][ff])
    out = {}
    for n in names:
        p = np.concatenate(pooled[n][0]) if pooled[n][0] else np.array([])
        q = np.concatenate(pooled[n][1]) if pooled[n][1] else np.array([])
        cases = [auc(np.concatenate(v[n][0]), np.concatenate(v[n][1])) for v in per_case.values()]
        cases = [a for a in cases if np.isfinite(a)]
        out[n] = dict(pooled_auc=auc(p, q), median_case_auc=float(np.median(cases)) if cases else np.nan,
                      cases=len(cases), ictal_median=float(np.median(p)) if len(p) else np.nan,
                      interictal_median=float(np.median(q)) if len(q) else np.nan,
                      _ictal=p, _inter=q)
    return out


# ── figures ─────────────────────────────────────────────────────────────────
def _style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(MUTED); ax.spines[s].set_linewidth(0.6)
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.yaxis.label.set_color(INK); ax.xaxis.label.set_color(INK)
    ax.grid(axis="y", color="#e6e5df", linewidth=0.5); ax.set_axisbelow(True)


def fig_per_case(scored_py, scored_fw, path):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    pc_py, pc_fw = scored_py["per_case"], scored_fw["per_case"]
    cases = sorted(pc_py)
    x = np.arange(len(cases)); w = 0.38
    fig, axs = plt.subplots(2, 1, figsize=(9, 5.2), sharex=True)
    for ax, key, lab in ((axs[0], "sens", "Sensitivity (%)"), (axs[1], "fa", "False alarms / h")):
        for off, pc, col, name in ((-w/2, pc_py, C_PY, "Python analyzer"), (w/2, pc_fw, C_FW, "ESP32 firmware")):
            v = [100*pc[c]["n_det"]/pc[c]["n_sz"] if key == "sens" else pc[c]["n_fa"]/pc[c]["hours"] for c in cases]
            ax.bar(x+off, v, w-0.04, color=col, label=name)
        ax.set_ylabel(lab, fontsize=9); _style(ax)
    axs[0].legend(frameon=False, fontsize=8, ncol=2, loc="upper right")
    axs[1].set_xticks(x); axs[1].set_xticklabels(cases, rotation=60, fontsize=7)
    fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)


def fig_features(fa, path):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    names = list(fa)
    fig, axs = plt.subplots(1, len(names), figsize=(11, 2.8))
    for ax, n in zip(axs, names):
        a, b = fa[n]["_inter"], fa[n]["_ictal"]
        if len(a) == 0 or len(b) == 0:
            ax.set_axis_off(); continue
        both = np.concatenate([a, b]); both = both[np.isfinite(both)]
        logx = n in ("Line-length", "Firmware abs(x−EMA)", "High-freq energy")
        if logx:
            pos = both[both > 0]
            lo, hi = np.percentile(pos, 0.5), np.percentile(pos, 99.5)
            bins = np.geomspace(max(lo, 1e-6), hi, 60); ax.set_xscale("log")
        else:
            lo, hi = np.percentile(both, 0.5), np.percentile(both, 99.5)
            bins = np.linspace(lo, hi, 60)
        ax.hist(a, bins, density=True, color="#9a9990", alpha=0.8, label="inter-ictal")
        ax.hist(b, bins, density=True, histtype="step", color=C_PY, linewidth=1.6, label="ictal")
        ax.set_title(f"{n}\nAUC {fa[n]['pooled_auc']:.2f}", fontsize=8.5, color=INK)
        ax.set_yticks([]); _style(ax); ax.grid(False)
    axs[0].legend(frameon=False, fontsize=7)
    fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)


def fig_example(rows, results, data_dir, channel, path):
    """One seizure: device-scale EEG + both detectors' state traces."""
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from chbmit import load_record, to_device_adc
    cand = [r for r in rows if r["channel"] == channel and r["seizures"]
            and (Path(data_dir) / f"{r['record']}.ch.json").exists()]
    if not cand:
        return False
    # prefer a seizure the Python analyzer detected
    def hit(r):
        on, off = r["seizures"][0]
        return any(on - 30 <= a <= off + 60 for a in r["py"]["seizure_alarms"])
    cand.sort(key=lambda r: (not hit(r), r["record"]))
    r = cand[0]; on, off = r["seizures"][0]
    rec = load_record(Path(data_dir), r["case"], r["record"], r["seizures"])
    adc = to_device_adc(rec.uv[channel], rec.fs)  # default front-end model
    z = np.load(results / "features" / f"{r['record']}_{channel}.npz")
    t0, t1 = max(0, on - 240), on + (off - on) + 240
    ts = np.arange(len(adc)) / 200.0; m = (ts >= t0) & (ts <= t1)
    fig, axs = plt.subplots(3, 1, figsize=(9, 4.6), sharex=True, gridspec_kw=dict(height_ratios=[2.2, 1, 1]))
    for ax in axs:
        ax.axvspan(on, off, color=SZ_FILL, zorder=0)
    axs[0].plot(ts[m], adc[m], color=INK, linewidth=0.35)
    axs[0].set_ylabel("ADC code", fontsize=8)
    axs[0].set_title(f"{r['record']} · {channel} · annotated seizure {on}–{off} s (shaded)", fontsize=9, color=INK)
    lab = ["NORMAL", "BROWN", "BLACK", "POST"]
    mt = (z["t"] >= t0) & (z["t"] <= t1)
    axs[1].step(z["t"][mt], z["state"][mt], where="post", color=C_PY, linewidth=1.2)
    mf = (z["fw_t10"] >= t0) & (z["fw_t10"] <= t1)
    axs[2].step(z["fw_t10"][mf], z["fw_state"][: len(z["fw_t10"])][mf], where="post", color=C_FW, linewidth=1.2)
    for ax, name in ((axs[1], "Python"), (axs[2], "Firmware")):
        ax.set_yticks(range(4)); ax.set_yticklabels(lab, fontsize=6.5); ax.set_ylabel(name, fontsize=8)
    axs[2].set_xlabel("time (s)", fontsize=8)
    for ax in axs:
        _style(ax)
    fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)
    return r["record"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", default="results")
    ap.add_argument("--data", default=None, help="data dir (for example-trace figure)")
    ap.add_argument("--channel", default="FP1-F7")
    a = ap.parse_args()
    res = Path(a.results)
    (res / "tables").mkdir(parents=True, exist_ok=True); (res / "figures").mkdir(exist_ok=True)
    rows = load(res)
    chans = sorted({r["channel"] for r in rows})

    summary, md = {}, []
    hdr = ("| Detector · alarm | Seizures detected | Mean per-patient sens. | False alarms | FA/h | FA/24 h | "
           "Latency median [IQR] (s) | Chance sens. at same alarm rate |\n|---|---|---|---|---|---|---|---|")
    total_h = {}
    for ch in chans:
        md.append(f"\n### Channel {ch}\n\n{hdr}")
        for det in ("py", "fw"):
            for kind in ("seizure_alarms", "any_alarms"):
                s, _ = evaluate(rows, det, kind, ch)
                s["chance"] = chance_sensitivity(rows, ch, s["fa_per_hour"])
                summary[f"{ch}|{det}|{kind}"] = {k: v for k, v in s.items() if k != "per_case"}
                md.append(fmt_row(f"{DET[det]} · {KIND[kind]}", s))
                total_h[ch] = s["hours"]
    cases = sorted({r["case"] for r in rows})
    n_rec = len({r["record"] for r in rows})
    head = (f"# Vera detector evaluation on CHB-MIT\n\n{len(cases)} patients · {n_rec} recordings · "
            f"{max(total_h.values()):.1f} h · scoring: alarm within [onset−30 s, offset+60 s] = detection; "
            f"alarms <90 s apart merged; all other alarms = false alarms.\n")
    (res / "tables" / "main_results.md").write_text(head + "\n".join(md) + "\n")

    # per-patient table (primary channel, BLACKOUT alarm)
    s_py, _ = evaluate(rows, "py", "seizure_alarms", a.channel)
    s_fw, _ = evaluate(rows, "fw", "seizure_alarms", a.channel)
    lines = ["case,seizures,hours,py_detected,py_fa,py_fa_per_h,fw_detected,fw_fa,fw_fa_per_h"]
    for c in sorted(s_py["per_case"]):
        p, f = s_py["per_case"][c], s_fw["per_case"][c]
        lines.append(f"{c},{p['n_sz']},{p['hours']:.2f},{p['n_det']},{p['n_fa']},{p['n_fa']/p['hours']:.2f},"
                     f"{f['n_det']},{f['n_fa']},{f['n_fa']/f['hours']:.2f}")
    (res / "tables" / f"per_patient_{a.channel}.csv").write_text("\n".join(lines) + "\n")
    fig_per_case(s_py, s_fw, res / "figures" / f"per_patient_{a.channel}.png")

    # artifact rejection & ADC usage
    art = {}
    for ch in chans:
        rr = [r for r in rows if r["channel"] == ch]
        art[ch] = dict(artifact_epochs=int(sum(r["py"]["artifact_epochs"] for r in rr)),
                       ictal_artifact_frac=float(sum(r["py"].get("ictal_artifact_ticks", 0) for r in rr) /
                                                 max(1, sum(r["py"].get("ictal_ticks", 0) for r in rr))),
                       interictal_artifact_frac=float(sum(r["py"].get("interictal_artifact_ticks", 0) for r in rr) /
                                                      max(1, sum(r["py"].get("interictal_ticks", 0) for r in rr))),
                       ticks=int(sum(r["py"]["ticks"] for r in rr)),
                       adc_clip_frac_mean=float(np.mean([r["adc_clip_frac"] for r in rr])),
                       time_in_state_py=list(np.mean([r["py"]["occupancy"] for r in rr], 0)),
                       time_in_state_fw=list(np.mean([r["fw"]["occupancy"] for r in rr], 0)))
    summary["signal_stats"] = art

    # feature discriminability
    fa = feature_analysis(rows, res, a.channel)
    summary["feature_auc"] = {k: {kk: vv for kk, vv in v.items() if not kk.startswith("_")} for k, v in fa.items()}
    fl = ["| Feature | Pooled AUC (ictal vs inter-ictal) | Median per-patient AUC | Ictal median | Inter-ictal median |",
          "|---|---|---|---|---|"]
    for k, v in fa.items():
        fl.append(f"| {k} | {v['pooled_auc']:.3f} | {v['median_case_auc']:.3f} (n={v['cases']}) | "
                  f"{v['ictal_median']:.3g} | {v['interictal_median']:.3g} |")
    (res / "tables" / f"feature_auc_{a.channel}.md").write_text("\n".join(fl) + "\n")
    fig_features(fa, res / "figures" / f"feature_distributions_{a.channel}.png")

    if a.data:
        ex = fig_example(rows, res, a.data, a.channel, res / "figures" / f"example_trace_{a.channel}.png")
        summary["example_record"] = ex

    summary["n_patients"] = len(cases); summary["n_recordings"] = n_rec
    summary["primary_channel"] = a.channel
    (res / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
    print((res / "tables" / "main_results.md").read_text())
    print("\n".join(fl))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {WORK}/evaluation/ablation.py
"""
Proposed revision: line-length-ratio detector (LL-R), evaluated with
patient-wise 2-fold cross-validation.

Motivation (from the feature analysis): line-length is the only one of the
three Python features that separates ictal from inter-ictal EEG well on a
single frontal channel; the 2-of-3 vote with DAR/HFE suppresses it.

Detector (all causal, ESP32-feasible — one running median of 300 values @1 Hz):
  LL1(t)  = mean of the analyzer's line-length over each 1 s (10 ticks)
  BG(t)   = median of LL1 over [t−330 s, t−30 s]      (lagged background)
  R(t)    = LL1(t) / BG(t)
  alarm   when R ≥ r for D consecutive seconds (no alarms during the first 330 s)
Grid: r × D.  Folds: odd vs even CHB-MIT case number (chb01 & chb21 — same
subject — stay together).  On the training fold pick the (r, D) with the
highest sensitivity whose FA/h ≤ target; report it on the held-out fold.

usage: python ablation.py --results results [--channel FP1-F7]
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from scoring import MERGE_S, score_record, summarise                   # noqa: E402

R_GRID = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 8.0]
D_GRID = [2, 4, 6, 8, 10, 15, 20]
BG_WIN, BG_LAG = 300, 30
FA_TARGETS = [1.0, 0.5, 0.2]


def ll_ratio(ft, ll, duration_s):
    n = int(np.floor(duration_s))
    sec = np.clip(ft.astype(np.int64), 0, n - 1)
    s = np.bincount(sec, weights=ll.astype(np.float64), minlength=n)
    c = np.bincount(sec, minlength=n)
    ll1 = np.where(c > 0, s / np.maximum(c, 1), np.nan)
    # forward-fill gaps (start-up)
    idx = np.where(np.isfinite(ll1), np.arange(n), 0); np.maximum.accumulate(idx, out=idx)
    ll1 = ll1[idx]
    import pandas as pd
    bg = pd.Series(ll1).rolling(BG_WIN, min_periods=BG_WIN // 2).median().shift(BG_LAG).to_numpy()
    r = ll1 / bg
    r[: BG_WIN + BG_LAG] = np.nan
    return r


def alarms_from_ratio(R, r, D):
    above = np.nan_to_num(R, nan=0.0) >= r
    # run length of consecutive 'above' seconds
    run = np.zeros(len(above), np.int32); cnt = 0
    for i, a in enumerate(above):
        cnt = cnt + 1 if a else 0
        run[i] = cnt
    onsets = np.where(run == D)[0].astype(float) + 1.0     # alarm at end of D-th second
    merged = []
    for a in onsets:
        if not merged or a - merged[-1] >= MERGE_S:
            merged.append(float(a))
    return merged


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", default="results")
    ap.add_argument("--channel", default="FP1-F7")
    a = ap.parse_args()
    res = Path(a.results)
    rows = [json.loads(l) for l in (res / "records.jsonl").read_text().splitlines() if l.strip()]
    rows = [r for r in rows if r["channel"] == a.channel and not r.get("missing")]

    ratios = {}
    for r in rows:
        z = np.load(res / "features" / f"{r['record']}_{a.channel}.npz")
        ratios[r["record"]] = ratio = ll_ratio(z["ft"], z["ll"], r["duration_s"])

    def run(rs, rr, dd):
        sc = []
        for r in rs:
            al = alarms_from_ratio(ratios[r["record"]], rr, dd)
            s = score_record(al, [tuple(x) for x in r["seizures"]], r["duration_s"])
            s["case"] = r["case"]; sc.append(s)
        return summarise(sc)

    grid = {}
    for rr in R_GRID:
        for dd in D_GRID:
            s = run(rows, rr, dd)
            grid[(rr, dd)] = s
    fold = lambda r: int(r["case"][3:]) % 2 if r["case"] != "chb21" else 1
    out = dict(channel=a.channel, grid=[dict(r=k[0], D=k[1], sensitivity=v["sensitivity"],
                                             fa_per_hour=v["fa_per_hour"], latency_median=v["latency_median"])
                                        for k, v in grid.items()], cv={})
    md = [f"# LL-ratio detector — patient-wise 2-fold CV ({a.channel})\n",
          "| FA/h target (train) | Fold → params | Held-out sens. | Held-out FA/h | Held-out latency median (s) |",
          "|---|---|---|---|---|"]
    for tgt in FA_TARGETS:
        test_rows_all, chosen = [], []
        for f in (0, 1):
            train = [r for r in rows if fold(r) != f]; test = [r for r in rows if fold(r) == f]
            if not train or not test:
                continue
            best = None
            for rr in R_GRID:
                for dd in D_GRID:
                    s = run(train, rr, dd)
                    if s["fa_per_hour"] <= tgt and (best is None or s["sensitivity"] > best[0] or
                                                    (s["sensitivity"] == best[0] and s["fa_per_hour"] < best[1])):
                        best = (s["sensitivity"], s["fa_per_hour"], rr, dd)
            if best is None:
                continue
            chosen.append((f, best[2], best[3]))
            for r in test:
                al = alarms_from_ratio(ratios[r["record"]], best[2], best[3])
                s = score_record(al, [tuple(x) for x in r["seizures"]], r["duration_s"])
                s["case"] = r["case"]; test_rows_all.append(s)
        if not test_rows_all:
            continue
        s = summarise(test_rows_all)
        out["cv"][str(tgt)] = dict(chosen=chosen, **{k: v for k, v in s.items() if k != "per_case"})
        md.append(f"| ≤ {tgt} | " + "; ".join(f"fold{f}: r={rr}, D={dd}s" for f, rr, dd in chosen) +
                  f" | {s['detected']}/{s['seizures']} ({100*s['sensitivity']:.1f}%) | {s['fa_per_hour']:.2f} | "
                  f"{s['latency_median']:.1f} |")
    (res / "tables").mkdir(exist_ok=True)
    (res / "tables" / f"llr_cv_{a.channel}.md").write_text("\n".join(md) + "\n")
    (res / f"llr_{a.channel}.json").write_text(json.dumps(out, indent=2, default=float))

    # operating-curve figure with the original detectors' points
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from report import evaluate, _style, C_PY, C_FW, INK
    fig, ax = plt.subplots(figsize=(5.6, 4))
    pts = np.array([(v["fa_per_hour"], 100 * v["sensitivity"]) for v in grid.values()])
    order = np.argsort(pts[:, 0]); front = []; best = -1
    for i in order:
        if pts[i, 1] > best:
            front.append(pts[i]); best = pts[i, 1]
    front = np.array(front)
    ax.scatter(pts[:, 0], pts[:, 1], s=9, color="#b9b8b0", label="LL-R grid (all r, D)")
    ax.plot(front[:, 0], front[:, 1], color=INK, linewidth=1.2, label="LL-R best trade-off")
    for det, col, name in (("py", C_PY, "Python analyzer (as shipped)"), ("fw", C_FW, "ESP32 firmware (as shipped)")):
        s, _ = evaluate(rows, det, "seizure_alarms", a.channel)
        ax.scatter([s["fa_per_hour"]], [100 * s["sensitivity"]], s=60, color=col, zorder=5,
                   edgecolor="white", linewidth=1.5, label=name)
    ax.set_xscale("symlog", linthresh=0.1); ax.set_xlabel("False alarms per hour", fontsize=9)
    ax.set_ylabel("Seizures detected (%)", fontsize=9); ax.set_ylim(0, 102)
    _style(ax); ax.legend(frameon=False, fontsize=7.5, loc="lower right")
    fig.tight_layout(); fig.savefig(res / "figures" / f"operating_curve_{a.channel}.png", dpi=200); plt.close(fig)
    print("\n".join(md))


if __name__ == "__main__":
    main()


In [ ]:
# ── 1. fetch CHB-MIT (Kaggle-side) and keep only FP1-F7 / FP2-F8 ────────────
t0 = time.time()
local = next((str(p) for p in Path("/kaggle/input").glob("*")), None) if Path("/kaggle/input").exists() else None
if not os.environ.get("VERA_DATA"):
    sh(f"{sys.executable} {WORK}/evaluation/fetch_chbmit.py --out {DATA} --workers 6"
       + (f" --local /kaggle/input" if local else "") + (" --all" if FETCH_ALL else ""))
n = len(list(Path(DATA).glob("*.ch.json")))
print(f"{n} recordings ready in {time.time()-t0:.0f}s;  disk used by extracted data:",
      sum(p.stat().st_size for p in Path(DATA).glob('*.ch.bin')) // 2**20, "MB")

In [ ]:
# ── 2. replay every recording through both detectors ───────────────────────
t0 = time.time()
sh(f"{sys.executable} {WORK}/evaluation/run_eval.py --data {DATA} --monitor {MONITOR} "
   f"--out {RESULTS}/records.jsonl --channels {' '.join(CHANNELS)} --jobs {JOBS} --gain {GAIN} --band {BAND}")
print(f"replay took {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── 3. tables, figures, feature analysis, cross-validated LL-ratio detector ─
for ch in [c for c in CHANNELS if c != PRIMARY] + [PRIMARY]:   # primary last → summary.json is the primary's
    sh(f"{sys.executable} {WORK}/evaluation/report.py --results {RESULTS} --data {DATA} --channel {ch}")
    shutil.copy(f"{RESULTS}/summary.json", f"{RESULTS}/summary_{ch}.json")
    sh(f"{sys.executable} {WORK}/evaluation/ablation.py --results {RESULTS} --channel {ch}")

In [ ]:
# ── 4. robustness: front-end gain sweep (primary channel) ──────────────────
sweep = {}
for g in GAIN_SWEEP:
    out = f"{WORK}/results_gain{int(g)}"
    sh(f"{sys.executable} {WORK}/evaluation/run_eval.py --data {DATA} --monitor {MONITOR} "
       f"--out {out}/records.jsonl --channels {PRIMARY} --jobs {JOBS} --gain {g} --band {BAND}")
    sh(f"{sys.executable} {WORK}/evaluation/report.py --results {out} --channel {PRIMARY}")
    sh(f"{sys.executable} {WORK}/evaluation/ablation.py --results {out} --channel {PRIMARY}")
    s = json.load(open(f"{out}/summary.json")); l = json.load(open(f"{out}/llr_{PRIMARY}.json"))
    sweep[g] = dict(py=s[f"{PRIMARY}|py|seizure_alarms"], fw=s[f"{PRIMARY}|fw|seizure_alarms"], llr_cv=l["cv"])
json.dump(sweep, open(f"{RESULTS}/gain_sweep.json", "w"), indent=2, default=float)
for g, v in sweep.items():
    print(f"gain {g:>6}: python sens {100*v['py']['sensitivity']:.1f}% @ {v['py']['fa_per_hour']:.2f} FA/h | "
          f"firmware sens {100*v['fw']['sensitivity']:.1f}% @ {v['fw']['fa_per_hour']:.2f} FA/h")

In [ ]:
# ── 5. package results ─────────────────────────────────────────────────────
import numpy, scipy, numba
manifest = dict(seizure_monitor_commit=commit, gain=GAIN, band=BAND, gain_sweep=GAIN_SWEEP, channels=CHANNELS,
                python=platform.python_version(), numpy=numpy.__version__, scipy=scipy.__version__,
                numba=numba.__version__, finished=time.strftime("%Y-%m-%d %H:%M:%S"),
                data="CHB-MIT Scalp EEG Database v1.0.0 (PhysioNet, doi:10.13026/C2K01R)")
json.dump(manifest, open(f"{RESULTS}/manifest.json", "w"), indent=2)
shutil.copytree(f"{WORK}/evaluation", f"{RESULTS}/code", dirs_exist_ok=True)
fl = Path(DATA) / "fetch_log.jsonl"
if fl.exists(): shutil.copy(fl, RESULTS)
small = Path(f"{WORK}/vera_results_tmp"); shutil.rmtree(small, ignore_errors=True)
shutil.copytree(RESULTS, small, ignore=shutil.ignore_patterns("features"))
shutil.make_archive(f"{WORK}/vera_results", "zip", small); shutil.rmtree(small)
shutil.make_archive(f"{WORK}/vera_features", "zip", f"{RESULTS}/features")
for z in ("vera_results.zip", "vera_features.zip"):
    print(z, os.path.getsize(f"{WORK}/{z}") // 2**10, "KB")
# keep the Output tab small: drop the raw extracted EEG
if os.path.isdir("/kaggle/working"): shutil.rmtree(DATA, ignore_errors=True)

In [ ]:
# ── 6. look at the results ─────────────────────────────────────────────────
from IPython.display import Markdown, Image, display
for f in ["main_results.md", f"feature_auc_{PRIMARY}.md"] + [f"llr_cv_{c}.md" for c in CHANNELS]:
    p = Path(RESULTS) / "tables" / f
    if p.exists(): display(Markdown(p.read_text()))
for f in sorted((Path(RESULTS) / "figures").glob("*.png")):
    display(Markdown(f"**{f.name}**")); display(Image(filename=str(f), width=820))